### Exercise 1

In [1]:
import torch
from torch import nn

X = torch.tensor([[0., 1., 2.],
                  [3., 4., 5.],
                  [6., 7., 8.]])

# Add batch and channel dimensions: (batch, channel, height, width)
X4 = X.reshape(1, 1, 3, 3)

# 2x2 average pooling through convolution
conv = nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=2,
    bias=False
)

# Uniform averaging kernel
conv.weight.data[:] = torch.ones((1, 1, 2, 2)) / 4

Y = conv(X4)
print(Y)

tensor([[[[2., 3.],
          [5., 6.]]]], grad_fn=<ConvolutionBackward0>)


### Exercise 2

Max-pooling cannot be implemented by convolution alone because convolution is **linear**, while max-pooling is **nonlinear**.

Take the simplest case: a $1 \times 2$ pooling window. Suppose convolution could implement max-pooling. Then there would be weights $w_1, w_2$ such that for all $a,b$,
$$
w_1 a + w_2 b = \max(a,b).
$$
Now plug in a few values.

First, let $(a,b)=(1,0)$:
$$
w_1 = \max(1,0)=1.
$$
Then let $(a,b)=(0,1)$:
$$
w_2 = \max(0,1)=1.
$$

So the convolution would have to be
$$
w_1 a + w_2 b = a+b.
$$

But now test $(a,b)=(1,1)$:
$$
a+b = 2,
$$

while
$$
\max(1,1)=1.
$$

Contradiction.

So no fixed convolution kernel can equal max-pooling for all inputs.


### Exercise 3

#### Part 1

Therefore,
$$
\max (a, b)=b+\operatorname{ReLU}(a-b)
$$
Equivalently, you could also write
$$
\max (a, b)=a+\operatorname{ReLU}(b-a)
$$


#### Part 2

In [5]:
import torch
import torch.nn.functional as F

def relu_max(a, b):
    return b + F.relu(a - b)

def max_pool2x2_relu(X):
    # X shape: (batch, channel, height, width)

    a = X[:, :, 0::2, 0::2]  # 0, 2, ... means starting from index 0, take every 2nd element
    b = X[:, :, 0::2, 1::2]
    c = X[:, :, 1::2, 0::2]
    d = X[:, :, 1::2, 1::2]

    m1 = relu_max(a, b)
    m2 = relu_max(c, d)

    return relu_max(m1, m2)

In [6]:
X = torch.tensor([[[[0., 1., 2., 3.],
                    [4., 5., 6., 7.],
                    [8., 9., 10., 11.],
                    [12., 13., 14., 15.]]]])

Y_relu = max_pool2x2_relu(X)
Y_builtin = F.max_pool2d(X, kernel_size=2, stride=2)

print(Y_relu)
print(Y_builtin)

tensor([[[[ 5.,  7.],
          [13., 15.]]]])
tensor([[[[ 5.,  7.],
          [13., 15.]]]])


#### Part 3

The key idea is:

* each ReLU block can compute a **pairwise max**
  $$
  \max(a,b)=b+\operatorname{ReLU}(a-b),
  $$
* so to compute max-pooling over many numbers, we build a **reduction tree** of pairwise maxima.

We should think in terms of:

1. how many intermediate feature maps (channels) we need,
2. how many sequential stages (layers) are needed.

---

**$2 \times 2$ max-pooling**

We must compute
$$
\max(a,b,c,d).
$$
A natural strategy:

**Layer 1: compute two pairwise maxima**
$$
m_1 = \max(a,b), m_2 = \max(c,d).
$$
These are two intermediate outputs, so we need:

* **2 channels**

at this stage.

---

**Layer 2: combine them**

Now compute
$$
\max(m_1,m_2).
$$
This produces the final pooled output.

So total:

* **2 ReLU/convolution stages (layers)**
* intermediate width of **2 channels**

---

**$3 \times 3$ max-pooling**

Now we must compute the max of 9 numbers:
$$
x_1,\dots,x_9.
$$
Each layer can only reduce by pairwise comparisons.

A balanced reduction looks like:

**Layer 1**

Compare pairs:
$$
(x_1,x_2),\ (x_3,x_4),\ (x_5,x_6),\ (x_7,x_8)
$$
giving 4 outputs, plus $x_9$ carried forward.

So after layer 1:

* 5 values remain
* need about **5 channels**

---

**Layer 2**

Reduce 5 values to about 3 values.

---

**Layer**

Reduce 3 values to 2 values.

---

**Layer 4**

Reduce 2 values to 1 value.

---

So roughly:

* **4 layers**
* maximum of about **5 channels**

if done efficiently.

---

**General pattern**

For $n$ numbers:

* pairwise max reduces the count roughly by half each layer,
* therefore depth is approximately
$$
\lceil \log_2 n \rceil.
$$
For a $k\times k$ pooling window:
$$
n=k^2.
$$
Hence:
$$
\boxed{\text{layers} \approx \lceil \log_2(k^2)\rceil}
$$
which is
$$
\boxed{\approx 2\log_2 k.}
$$
And the number of channels is proportional to the number of intermediate maxima being stored during the reduction tree.


### Exercise 4

What is the computational cost of the pooling layer? Assume that the input to the pooling layer is of size $c \times h \times w$, the pooling window has a shape of $p_{\mathrm{h}} \times p_{\mathrm{w}}$ with a padding of $\left(p_{\mathrm{h}}, p_{\mathrm{w}}\right)$ and a stride of $\left(s_{\mathrm{h}}, s_{\mathrm{w}}\right)$.

For pooling, the cost is:
$$
\text{number of output entries}
\times
\text{cost per pooling window}.
$$
The lecture note emphasizes that pooling applies independently to each channel, so the output keeps $c$ channels. 

The output height and width are approximately:
$$
h_{\text{out}} =  \left\lfloor
\frac{h + 2p_{\mathrm{h}} - p_{\mathrm{h}}}{s_{\mathrm{h}}}
\right\rfloor + 1 = \left\lfloor
\frac{h + p_{\mathrm{h}}}{s_{\mathrm{h}}}
\right\rfloor + 1
$$
$$ 
w_{\text{out}} = \left\lfloor
\frac{w + 2p_{\mathrm{w}} - p_{\mathrm{w}}}{s_{\mathrm{w}}}
\right\rfloor + 1 =  \left\lfloor
\frac{w + p_{\mathrm{w}}}{s_{\mathrm{w}}}
\right\rfloor + 1
]
$$
because the padding is $(p_{\mathrm{h}}, p_{\mathrm{w}})$, meaning $p_{\mathrm{h}}$ on both top/bottom and $p_{\mathrm{w}}$ on both left/right.

Each output value pools over a window of size
$$
p_{\mathrm{h}}p_{\mathrm{w}}.
$$
So the total computational cost is:
$$
\boxed{
O\left(
c
\cdot
h_{\text{out}}
\cdot
w_{\text{out}}
\cdot
p_{\mathrm{h}}
p_{\mathrm{w}}
\right)
}
$$
Substituting the output dimensions:
$$
\boxed{
O\left(
c
\left(
\left\lfloor\frac{h+p_{\mathrm{h}}}{s_{\mathrm{h}}}\right\rfloor+1
\right)
\left(
\left\lfloor\frac{w+p_{\mathrm{w}}}{s_{\mathrm{w}}}\right\rfloor+1
\right)
p_{\mathrm{h}}p_{\mathrm{w}}
\right)
}
$$
For max-pooling, this counts comparisons. For average pooling, this counts additions plus one division per output.


### Exercise 5

The key difference is:

* **average pooling aggregates all information**
* **max-pooling selects the strongest activation**

These lead to very different behaviors.

---

**1. Average pooling behaves like smoothing**
Average pooling computes
$$
\frac{1}{p_hp_w}\sum x_{ij}.
$$
So every value inside the window contributes.

Intuition:

* acts like a blur/downsampling filter,
* preserves overall background statistics,
* reduces noise by averaging nearby values.

This is similar to classical signal processing:
averaging nearby pixels smooths the image.

But this also means:

* strong features can get diluted,
* edges/textures become weaker.

Example:

Suppose a feature detector fires strongly at one location:
$$
\begin{bmatrix}
0 & 0 \\
0 & 10
\end{bmatrix}
$$
Average pooling gives
$$
2.5.
$$
The strong signal gets weakened.

---

**2. Max-pooling behaves like feature selection**

Max-pooling computes
$$
\max x_{ij}.
$$
Only the strongest activation matters.

Intuition:

* asks:
  “Did this feature appear anywhere in this region?”
* preserves strong activations,
* discards weaker/background responses.

Using the same example:
$$
\begin{bmatrix}
0 & 0 \\
0 & 10
\end{bmatrix}
$$

Max-pooling gives
$$
10.
$$
So the detector response survives intact.

**3. Information retained**

Average pooling preserves:

* overall intensity/statistics,
* coarse global information.

Max-pooling preserves:

* existence of salient features,
* strongest evidence.

This is why CNNs historically preferred max-pooling for recognition tasks:

object recognition often depends more on whether a feature exists than on its exact average strength.


### Exercise 6

We do not really need a separate minimum-pooling layer because minimum pooling can be expressed using max-pooling.

The key identity is:
$$
\boxed{
\min(x_1,\dots,x_n) = -\max(-x_1,\dots,-x_n)
}
$$
So minimum pooling is equivalent to:

1. negate the input,
2. apply max-pooling,
3. negate again.

---

For a pooling window:
$$
X =
\begin{bmatrix}
1 & 5 \\
2 & 3
\end{bmatrix}
$$
Minimum pooling gives:
$$
\min = 1.
$$
Now negate:
$$
-X =
\begin{bmatrix}
-1 & -5 \\
-2 & -3
\end{bmatrix}
$$
Max-pooling on $-X$:
$$
\max(-1,-5,-2,-3)=-1.
$$
Negate again:
$$
-(-1)=1.
$$
Same result.

---

So in practice:
$$
\boxed{
\text{MinPool}(X) = -\text{MaxPool}(-X)
}
$$
This means a framework only really needs max-pooling as a primitive.

---

Why is max-pooling preferred anyway?

In CNNs, large activations usually represent:

* strong feature detections,
* high confidence,
* salient edges/textures.

Max-pooling preserves these strong responses.

Minimum pooling would instead preserve the *least* activated feature, which is usually not useful for recognition tasks. It tends to emphasize absence/background/noise rather than salient structure.

That is why max-pooling became standard while min-pooling is rarely used.


### Exercise 7

Softmax pooling is possible, but it sits somewhere between average pooling and max-pooling, and usually does not give enough benefits to justify the extra complexity.

The idea would be:
$$
y = \sum_i
\frac{e^{x_i}}{\sum_j e^{x_j}}
x_i.
$$

So instead of:

* averaging all values equally (average pooling),
* or selecting only the maximum (max-pooling),

softmax pooling computes a **weighted average**, where larger activations receive exponentially larger weights.

---

**High-level intuition**

Softmax pooling behaves like a “soft maximum.”

If one activation is much larger than the others:
$$
x_k \gg x_i,
$$
then
$$
\frac{e^{x_k}}{\sum_j e^{x_j}} \approx 1,
$$
so softmax pooling becomes close to max-pooling.

If all activations are similar, it behaves more like average pooling.

So it interpolates between the two.

---

**Why might it not be popular?**

**1. More computationally expensive**

Average pooling:

* additions + division.

Max-pooling:

* comparisons only.

Softmax pooling requires:

* exponentials,
* sums,
* divisions.

Exponentials are relatively expensive operations, especially historically on GPUs/CPUs.

So the computational cost is much higher.

---

**2. Less numerically stable**

Softmax can overflow:
$$
e^{1000}
$$

is huge.

So implementations need stabilization tricks like subtracting the max:
$$
e^{x_i-\max_j x_j}.
$$

Pooling layers are supposed to be simple and cheap. Softmax pooling introduces extra numerical concerns.
